# 6-1. 미니 GPT — 부품을 모아 완성하기

> **이 노트북에서 할 일**
> 5장에서 만든 부품으로 **끝까지 작동하는 GPT**를 조립하고, 학습시키고, 글을 쓰게 합니다.

## 구조 전체도

```
토큰 인덱스 (B, T)
      │
  토큰 임베딩 + 위치 임베딩          (B, T, C)
      │
  ┌───┴────────────────┐
  │  Block x n_layer   │   ← 5-2에서 만든 그것
  │   x = x + attn(ln(x))
  │   x = x + ffwd(ln(x))
  └───┬────────────────┘
      │
  최종 LayerNorm
      │
  Linear(C -> vocab_size)           (B, T, vocab_size)
      │
   다음 토큰 확률
```

4-1의 NPLM과 비교하면 달라진 점은 단 두 가지입니다.
**(1) 은닉층이 트랜스포머 블록으로 바뀌었고, (2) 모든 위치에서 동시에 예측합니다.**

### (2)가 중요합니다 — 학습 효율의 차이

NPLM은 `(앞 3단어) → 다음 1단어` 한 쌍을 학습에 썼습니다.
GPT는 길이 T짜리 시퀀스 하나에서 **T개의 예측을 동시에** 만듭니다.

```
입력 : 언 어  모 델 은  다
정답 :  어 모  델 은 다  음
       ↑  ↑   ↑  ↑  ↑   ↑     한 번에 6개를 학습
```

같은 데이터로 T배 많이 배우는 셈입니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import load_corpus, set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

## 1. 데이터 — 글자 단위

4-2에서 본 대로, 작은 말뭉치에서는 글자 단위가 안전합니다(`<unk>` 가 없음).

In [ ]:
# 여기에 진짜 한국어 텍스트 파일 경로를 넣으면 그대로 돌아갑니다 (4-2 참고)
CORPUS_PATH = None
text = load_corpus(CORPUS_PATH)

chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
vocab_size = len(chars)

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

print(f"전체 {len(data):,} 글자,  어휘 {vocab_size}자")
print(f"학습 {len(train_data):,} / 평가 {len(val_data):,}")
print(f"{chr(10)}인코딩 확인: {encode('언어 모델')} -> {decode(encode('언어 모델'))!r}")

In [ ]:
block_size = 64      # 문맥 길이 (한 번에 보는 글자 수)
batch_size = 32

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])    # 한 칸 밀린 것이 정답
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print(f"x {tuple(xb.shape)}, y {tuple(yb.shape)}")
print(f"{chr(10)}입력: {decode(xb[0, :20].tolist())!r}")
print(f"정답: {decode(yb[0, :20].tolist())!r}   <- 한 글자씩 밀려 있습니다")

## 2. 모델

5-2에서 만든 부품을 그대로 가져옵니다. 이번엔 `F.scaled_dot_product_attention` 을 써서
멀티헤드를 **한 번의 행렬 연산**으로 처리합니다 (헤드마다 반복문을 도는 것보다 훨씬 빠릅니다).

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_embd = n_head, n_embd
        self.attn = nn.Linear(n_embd, 3 * n_embd, bias=False)   # Q,K,V 를 한 번에
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)
        self.p = dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.attn(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))


class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4,
                 block_size=block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        # 가중치 공유(weight tying): 입력 임베딩과 출력층이 같은 행렬을 씁니다.
        # 파라미터가 줄고 성능도 보통 좋아집니다. GPT-2 부터의 표준.
        self.head.weight = self.tok_emb.weight

        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"문맥이 {self.block_size} 를 넘었습니다"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))               # (B, T, vocab_size)

        if targets is None:
            return logits, None
        # (B,T,V) -> (B*T,V) 로 펴서 한 번에 손실 계산
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=200, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]        # 문맥 길이를 넘으면 앞을 자름
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature      # 마지막 위치만 사용
            if top_k is not None:
                v, _ = logits.topk(min(top_k, logits.size(-1)))
                logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))
            idx_next = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
        self.train()
        return idx

set_seed(1234)
model = MiniGPT(vocab_size).to(device)
print(f"파라미터: {human(count_params(model))}")
print(f"기준선: 학습 전 손실 = log(V) = {math.log(vocab_size):.3f}")

logits, loss = model(*get_batch("train"))
print(f"실제 초기 손실 = {loss.item():.3f}   <- 기준선과 비슷해야 정상입니다")

> **초기 손실 점검은 중요한 습관입니다.**
> $\log V$ 와 크게 다르면 초기화가 잘못된 것입니다.
> 예를 들어 손실이 10 이상이면 가중치가 너무 크게 초기화되어 학습이 힘들어집니다.

## 3. 학습

In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=20):
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = torch.zeros(iters)
        for k in range(iters):
            _, l = model(*get_batch(split))
            losses[k] = l.item()
        out[split] = losses.mean().item()
    model.train()
    return out

max_steps, eval_every = 1500, 250
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1,
                        betas=(0.9, 0.95))
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=max_steps + 1,
                                            pct_start=0.1)

t0 = time.perf_counter()
history = []
for step in range(max_steps + 1):
    if step % eval_every == 0:
        L = estimate_loss(model)
        history.append((step, L["train"], L["val"]))
        print(f"step {step:5d} | train {L['train']:.4f} (ppl {math.exp(L['train']):7.2f})"
              f" | val {L['val']:.4f} (ppl {math.exp(L['val']):8.2f})"
              f" | lr {sched.get_last_lr()[0]:.2e}")
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # 기울기 폭발 방지
    opt.step(); sched.step()

print(f"{chr(10)}학습 시간: {time.perf_counter()-t0:.1f}초 ({device})")

### 학습 루프에 새로 들어간 것들

| 기법 | 하는 일 | 왜 |
|---|---|---|
| `weight_decay=0.1` | 가중치를 0쪽으로 당김 | 과적합 완화 |
| `betas=(0.9, 0.95)` | Adam 2차 모멘텀을 낮춤 | LLM 학습의 관례 |
| `OneCycleLR` | 학습률을 올렸다 내림 | **워밍업**이 초기 발산을 막음 |
| `clip_grad_norm_` | 기울기 크기를 1.0으로 제한 | 드물게 튀는 배치가 모델을 망치는 걸 방지 |

**워밍업**은 특히 중요합니다. 학습 초반엔 가중치가 무작위라 기울기가 엉망인데,
거기에 큰 학습률을 곱하면 모델이 망가집니다. 그래서 작게 시작해 서서히 올립니다.
실제 LLM 사전학습 설정에는 예외 없이 들어 있습니다.

In [ ]:
import matplotlib.pyplot as plt

steps = [h[0] for h in history]
plt.figure(figsize=(7, 4))
plt.plot(steps, [h[1] for h in history], "o-", label="train")
plt.plot(steps, [h[2] for h in history], "s-", label="val")
plt.axhline(math.log(vocab_size), ls=":", c="red",
            label=f"baseline log(V)={math.log(vocab_size):.2f}")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3)
plt.title("MiniGPT 학습 곡선")
plt.tight_layout(); plt.show()

## 4. 생성

In [ ]:
ctx = torch.tensor([encode("언어 모델은")], device=device)
for T in (0.5, 0.8, 1.2):
    out = model.generate(ctx, max_new_tokens=150, temperature=T, top_k=20)
    print(f"--- 온도 {T} ---")
    print(decode(out[0].tolist()))
    print()

### 결과를 어떻게 볼 것인가

우리 말뭉치는 1,500자 남짓입니다. **모델이 통째로 외우고 있을 가능성이 매우 높습니다.**
val 손실이 train 손실보다 한참 높다면 그게 증거입니다.

그래도 확인할 가치가 있는 것들이 있습니다.

- **한글 글자를 조합해서 냈는가** (깨진 문자가 아니라)
- **띄어쓰기 자리가 그럴듯한가**
- **문장이 마침표로 끝나는가**

이것만 해도 모델이 "글자 수준의 한국어 구조"를 잡았다는 뜻입니다.
NPLM이 단어를 뒤섞어 놓기만 했던 것과 비교해 보세요.

### 외운 것인지 확인하기

In [ ]:
set_seed(0)
gen = decode(model.generate(torch.tensor([encode("언어")], device=device),
                            max_new_tokens=300, temperature=0.8, top_k=20)[0].tolist())

# 생성문에서 길이 20짜리 조각을 뽑아 원문에 그대로 있는지 확인
chunks = [gen[i:i+20] for i in range(0, len(gen) - 20, 20)]
copied = sum(1 for c in chunks if c in text)
print(f"20글자 조각 {len(chunks)}개 중 원문에 그대로 있는 것: {copied}개 ({copied/len(chunks):.0%})")
print(f"{chr(10)}높다면 = 외운 것입니다. 데이터를 늘리면 이 비율이 떨어집니다.")

## 5. 규모를 키우면 — 스케일링 감각

모델을 키우면 무엇이 얼마나 드는지 미리 계산해 두는 건 실무의 기본입니다.

In [ ]:
def gpt_params(n_layer, n_embd, vocab, block):
    emb = vocab * n_embd + block * n_embd            # 토큰 + 위치 (출력층은 공유)
    per_block = 4 * n_embd * n_embd + 8 * n_embd * n_embd   # attn(QKV+proj) + ffn(4x 두 개)
    return emb + n_layer * per_block

print(f"{'구성':>28} | {'파라미터':>10} | {'fp16 가중치':>12} | {'AdamW 학습 메모리':>18}")
print("-" * 80)
configs = [
    ("우리 MiniGPT (4층,128)", 4, 128, vocab_size, 64),
    ("GPT-2 small (12층,768)", 12, 768, 50257, 1024),
    ("GPT-2 medium (24층,1024)", 24, 1024, 50257, 1024),
    ("GPT-2 XL (48층,1600)", 48, 1600, 50257, 1024),
    ("Llama-2 7B (32층,4096)", 32, 4096, 32000, 4096),
]
for name, L, E, V_, B_ in configs:
    p = gpt_params(L, E, V_, B_)
    w = p * 2 / 1e9
    # AdamW: 가중치(4) + 기울기(4) + m(4) + v(4) = 파라미터당 약 16바이트
    train_mem = p * 16 / 1e9
    print(f"{name:>28} | {human(p):>10} | {w:>9.2f} GB | {train_mem:>15.1f} GB")

print(f"{chr(10)}주목: 학습에 필요한 메모리는 가중치의 약 8배입니다(옵티마이저 상태 때문).")
print("7B 모델을 통째로 학습하려면 100GB 이상 -- 개인 GPU로는 불가능합니다.")
print("이것이 6-2의 클라우드 스케일업, 그리고 파인튜닝에서 LoRA를 쓰는 이유입니다.")

---

## 정리

**완성한 것**

```python
x = tok_emb(idx) + pos_emb(pos)
for block in blocks:
    x = x + attn(ln1(x))
    x = x + ffwd(ln2(x))
logits = head(ln_f(x))
```

이것이 GPT의 전부입니다. **정말로 전부입니다.**
GPT-4도 이 구조에서 층과 차원을 키우고 5-3의 현대적 부품을 넣은 것입니다.

**남은 것은 규모뿐입니다.**

| | 우리 | GPT-3 |
|---|---|---|
| 파라미터 | 수십만 | 175,000,000,000 |
| 학습 데이터 | 1.5KB | 약 570GB |
| 학습 비용 | 수십 초 | 수백만 달러 |

## 연습

1. `n_layer`, `n_embd` 를 키워 보세요. 언제부터 M1에서 느려지나요?
2. `block_size` 를 32 / 128 / 256 으로 바꿔 학습 시간과 손실을 비교하세요.
3. 4-2에서 만든 큰 한국어 말뭉치 파일 경로를 `CORPUS_PATH` 에 넣고 다시 돌려 보세요.
   **5절의 "외운 비율"이 확 떨어지는 순간이 진짜 언어 모델이 되는 순간입니다.**

## 다음

→ `0602_ScaleUp_RunPod.md` : 이 코드를 클라우드 GPU에서 제대로 돌리기